In [3]:
#!/usr/bin/env python3
"""
seinfeld_wiki_scrape.py

Extracts all Seinfeld episode rows (seasons 1–9) from Wikipedia's
'List of Seinfeld episodes' page, including short descriptions when available.

Outputs:
  - seinfeld_episodes.csv
  - seinfeld_episodes.jsonl

Source page:
  https://en.wikipedia.org/wiki/List_of_Seinfeld_episodes

LICENSE / ATTRIBUTION
---------------------
Wikipedia content is available under the Creative Commons Attribution-ShareAlike License (CC BY-SA 4.0).
If you publish or share the extracted dataset, you MUST:
  1) Provide attribution and a link back to the source page (and ideally the edit history).
  2) Distribute your dataset under CC BY-SA 4.0 (share-alike).
Suggested attribution to include alongside the files:
  "Episode metadata and short descriptions derived from Wikipedia’s
   'List of Seinfeld episodes' (CC BY-SA 4.0). © Wikipedia contributors."
"""

import csv
import json
import re
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup

WIKI_URL = "https://en.wikipedia.org/wiki/List_of_Seinfeld_episodes"

# If you later want to follow each episode page to get longer plot sections,
# set this to True (be respectful and rate-limit). The list page usually
# has short synopses already; this scraper focuses on that page only.
FOLLOW_EPISODE_PAGES = False
REQUEST_DELAY = 0.5  # seconds between requests if FOLLOW_EPISODE_PAGES = True

HEADERS = {
    "User-Agent": "Research bot for academic use (contact: your-email@example.com)"
}

def normalize(text):
    if text is None:
        return ""
    return re.sub(r"\s+", " ", text).strip()

def extract_season_from_header(header_tag):
    """
    Given the header above an episode table (usually <h2> or <h3>),
    return something like 'Season 4' or 'Season 4 (1992–93)'.
    """
    if not header_tag:
        return ""
    title_text = normalize(header_tag.get_text())
    # Typically the header looks like: "Season 1 (1990)"
    # We'll keep the text as-is.
    return title_text

def find_season_header(table):
    """
    Walk upward to find the nearest preceding h2/h3 header (season title).
    """
    node = table
    while node:
        node = node.find_previous(["h2", "h3"])
        if node and node.name in ("h2", "h3"):
            # Skip 'Series overview' etc.
            if "Season" in node.get_text():
                return node
            # keep searching if we hit unrelated headers
    return None

def parse_episode_list_page(url=WIKI_URL):
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    # Episode tables typically have class 'wikiepisodetable'
    tables = soup.select("table.wikiepisodetable")
    items = []

    for t in tables:
        # identify season from preceding header
        season_hdr = find_season_header(t)
        season_text = extract_season_from_header(season_hdr)

        # Rows that represent episodes often have class 'vevent'
        for row in t.select("tr.vevent"):
            cols = row.find_all(["th", "td"])
            if len(cols) < 3:
                continue

            # Wikipedia episode list tables usually follow a pattern:
            # [0]=No.overall, [1]=No. in season, [2]=Title, [3]=Directed by,
            # [4]=Written by, [5]=Original air date, [6]=Prod. code, [7]=U.S. viewers (varies by season)
            # A short description may appear in a dedicated column (often last), or be absent.
            def col_text(i):
                return normalize(cols[i].get_text()) if i < len(cols) else ""

            # Extract basics
            overall_no = col_text(0)
            season_no = col_text(1)

            # Title and link
            title = ""
            title_link = ""
            if len(cols) >= 3:
                # title can be inside quotes / italics
                title_cell = cols[2]
                title = normalize(title_cell.get_text())
                a = title_cell.find("a")
                if a and a.get("href", "").startswith("/wiki/"):
                    title_link = "https://en.wikipedia.org" + a["href"]

            directed_by = col_text(3) if len(cols) > 3 else ""
            written_by = col_text(4) if len(cols) > 4 else ""

            # Original air date sometimes has a <span class="bday dtstart published updated">
            air_date = ""
            if len(cols) > 5:
                # try to pull machine-readable date if present
                date_span = cols[5].find("span", {"class": "bday dtstart published updated"})
                if not date_span:
                    date_span = cols[5].find("span", {"class": "dtstart"})
                air_date = normalize(date_span.get_text()) if date_span else col_text(5)

            prod_code = col_text(6) if len(cols) > 6 else ""
            us_viewers = col_text(7) if len(cols) > 7 else ""

            # Short summary/description — this column isn't always present or may have a specific class.
            # We’ll try common selectors, then fall back to empty.
            short_summary = ""
            # common classes used by Wikipedia episode tables:
            desc_cell = row.find("td", class_="description")
            if desc_cell:
                short_summary = normalize(desc_cell.get_text())
            else:
                # fallback: sometimes the last column is a synopsis
                # but make sure we don't just grab viewers column; prefer descriptive text
                candidates = [c for c in cols[3:] if len(normalize(c.get_text())) > 40]
                if candidates:
                    short_summary = normalize(candidates[-1].get_text())

            # Optionally fetch episode page for a longer plot
            long_plot = ""
            if FOLLOW_EPISODE_PAGES and title_link:
                try:
                    time.sleep(REQUEST_DELAY)
                    ep = requests.get(title_link, headers=HEADERS, timeout=30)
                    ep.raise_for_status()
                    ep_soup = BeautifulSoup(ep.text, "html.parser")

                    # Try to find 'Plot' or 'Synopsis' section text
                    # Strategy: find the header and collect following paragraphs until next header
                    def section_text(names=("Plot", "Synopsis")):
                        for hdr in ep_soup.select("h2, h3"):
                            ht = hdr.get_text().strip()
                            for n in names:
                                if ht.startswith(n):
                                    buf = []
                                    sib = hdr.find_next_sibling()
                                    while sib and sib.name not in ("h2", "h3"):
                                        if sib.name == "p":
                                            buf.append(sib.get_text(" ", strip=True))
                                        sib = sib.find_next_sibling()
                                    return " ".join(buf)
                        return ""
                    long_plot = section_text()
                except Exception:
                    long_plot = ""

            items.append({
                "season": season_text,
                "overall_no": overall_no,
                "season_no": season_no,
                "title": title.strip('"'),
                "title_link": title_link,
                "directed_by": directed_by,
                "written_by": written_by,
                "original_air_date": air_date,
                "prod_code": prod_code,
                "us_viewers_millions": us_viewers,
                "short_summary": short_summary,
                "long_plot_optional": long_plot,
                "source_page": WIKI_URL,
            })

    return items

def write_outputs(rows, out_csv="seinfeld_episodes.csv", out_jsonl="seinfeld_episodes.jsonl"):
    Path(".").mkdir(parents=True, exist_ok=True)
    # CSV
    fieldnames = [
        "season","overall_no","season_no","title","title_link",
        "directed_by","written_by","original_air_date","prod_code","us_viewers_millions",
        "short_summary","long_plot_optional","source_page"
    ]
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)
    # JSONL
    with open(out_jsonl, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def main():
    rows = parse_episode_list_page(WIKI_URL)
    # Sanity: filter empties and keep obvious episodes
    rows = [r for r in rows if r.get("title")]
    print(f"Collected {len(rows)} episode rows.")
    # Expect ~180
    write_outputs(rows)
    print("Wrote: seinfeld_episodes.csv and seinfeld_episodes.jsonl")
    print("\nAttribution to include with the files:")
    print('Episode metadata and short descriptions derived from Wikipedia’s "List of Seinfeld episodes" '
          '(https://en.wikipedia.org/wiki/List_of_Seinfeld_episodes), CC BY-SA 4.0. © Wikipedia contributors.')

if __name__ == "__main__":
    main()


Collected 180 episode rows.
Wrote: seinfeld_episodes.csv and seinfeld_episodes.jsonl

Attribution to include with the files:
Episode metadata and short descriptions derived from Wikipedia’s "List of Seinfeld episodes" (https://en.wikipedia.org/wiki/List_of_Seinfeld_episodes), CC BY-SA 4.0. © Wikipedia contributors.


In [5]:
#!/usr/bin/env python3
# seinfeld_fetch_plots_v5.py — robust Plot extractor + safe JSONL handling

import csv, json, re, time
from pathlib import Path
from typing import List, Optional
import requests
from bs4 import BeautifulSoup, Tag

INPUT_JSONL = "seinfeld_episodes.jsonl"
OUT_PLOTS_JSONL = "seinfeld_plots.jsonl"
OUT_MERGED_CSV = "seinfeld_episodes_with_plots.csv"

# Set to True to re-fetch EVERY url regardless of prior status.
FORCE = False

HEADERS = {
    "User-Agent": "SeinfeldPlotBot/1.0 (+you@example.org) requests",
    "Accept-Language": "en",
}
REQUEST_DELAY = 0.6
MAX_RETRIES, RETRY_SLEEP = 3, 2

START_IDS = (
    "Plot", "Synopsis", "Summary", "Story",
    "Plot_summary", "Plot_summary_and_analysis"
)

STOP_IDS = (
    "Production", "Cast", "Themes", "Reception", "Legacy", "Notes",
    "References", "External_links", "Cultural_references", "Home_media",
    "Music", "Credits", "See_also", "Further_reading", "Analysis"
)

# ------------------------------------------------------------
# Utility helpers
# ------------------------------------------------------------

def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

def _strip_refs(text: str) -> str:
    return re.sub(r"\s*\[[^\]]+\]", "", text)

def _clean(text: str) -> str:
    return _norm(_strip_refs(text))

def _tokey(s: str) -> str:
    return (s or "").strip().lower().replace(" ", "_")

def _same_id(a: str, b: str) -> bool:
    return _tokey(a) == _tokey(b)

# ------------------------------------------------------------
# HTML parsing helpers
# ------------------------------------------------------------

def _find_header_node(soup: BeautifulSoup, ids: tuple) -> Optional[Tag]:
    # 1) exact id on mw-headline span
    for sid in ids:
        el = soup.select_one(f'span.mw-headline[id="{sid}"]')
        if el:
            return el.find_parent(["h2", "h3"]) or el
    # 2) exact id on any h2/h3 (mobile/variant)
    for sid in ids:
        el = soup.select_one(f'h2#{sid}, h3#{sid}')
        if el:
            return el
    # 3) text startswith on mw-headline span
    for el in soup.select("span.mw-headline"):
        txt = _norm(el.get_text())
        if any(txt.lower().startswith(s.lower()) for s in ids):
            return el.find_parent(["h2", "h3"]) or el
    # 4) text startswith on h2/h3
    for el in soup.select("h2, h3"):
        txt = _norm(el.get_text())
        if any(txt.lower().startswith(s.lower()) for s in ids):
            return el
    return None


def _collect_between_headers(start_header: Tag, stop_ids: tuple) -> str:
    header = start_header if start_header.name in ("h2", "h3") else start_header.find_parent(["h2","h3"])
    if not header:
        return ""
    paras = []
    node = header.next_sibling
    while node:
        if isinstance(node, Tag) and node.name in ("h2", "h3"):
            headline = node.select_one(".mw-headline")
            hid = (headline and (headline.get("id") or headline.get_text())) or node.get("id") or node.get_text()
            if any(_same_id(hid, sid) for sid in stop_ids):
                break
            break
        if isinstance(node, Tag) and node.name == "p":
            for sup in node.select("sup.reference"):
                sup.decompose()
            txt = _clean(node.get_text(" ", strip=True))
            if txt:
                paras.append(txt)
        node = node.next_sibling
    return _clean(" ".join(paras))


def _lead_fallback(soup: BeautifulSoup) -> str:
    root = soup.select_one("#mw-content-text .mw-parser-output")
    if not root:
        return ""
    lead = []
    for child in root.children:
        if isinstance(child, Tag) and child.name == "p":
            t = _clean(child.get_text(" ", strip=True))
            if len(t) > 60:
                lead.append(t)
            if len(lead) >= 2:
                break
        if isinstance(child, Tag) and (child.get("id") == "toc" or child.name in ("h2","h3")):
            break
    return _clean(" ".join(lead))

# ------------------------------------------------------------
# Fetch logic with retries
# ------------------------------------------------------------

def fetch_plot(url: str) -> str:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30, allow_redirects=True)
            if r.status_code == 429:
                time.sleep(RETRY_SLEEP * attempt)
                continue
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")
            start_header = _find_header_node(soup, START_IDS)
            plot = _collect_between_headers(start_header, STOP_IDS) if start_header else ""
            if not plot:
                plot = _lead_fallback(soup)
            return plot
        except Exception as exc:
            print(f"[WARN] Fetch attempt {attempt} failed for {url}: {exc}")
            if attempt == MAX_RETRIES:
                return ""
            time.sleep(RETRY_SLEEP * attempt)
    return ""

# ------------------------------------------------------------
# File I/O utilities
# ------------------------------------------------------------

def read_jsonl(path: str) -> List[dict]:
    """Safe JSONL reader that skips malformed lines."""
    rows = []
    p = Path(path)
    if not p.exists():
        return rows
    with open(p, "r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            s = line.strip()
            if not s:
                continue
            try:
                obj = json.loads(s)
                if isinstance(obj, dict):
                    rows.append(obj)
                else:
                    print(f"[WARN] {path}:{lineno} — not a dict; skipped.")
            except json.JSONDecodeError as e:
                print(f"[WARN] {path}:{lineno} — bad JSON: {e}. Skipped.")
                continue
    return rows


def append_jsonl(path: str, rec: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def already_done_index(path: str) -> dict:
    done = {}
    for rec in read_jsonl(path):
        key = rec.get("title_link") or rec.get("url")
        if key:
            done[key] = rec   # last one wins
    return done

# ------------------------------------------------------------
# Optional cleaning utility
# ------------------------------------------------------------

def clean_jsonl(src: str, dst: str):
    """Write only valid JSON lines to a new file."""
    good, bad = 0, 0
    with open(src, "r", encoding="utf-8") as fin, open(dst, "w", encoding="utf-8") as fout:
        for lineno, line in enumerate(fin, 1):
            s = line.strip()
            if not s:
                continue
            try:
                obj = json.loads(s)
                if not isinstance(obj, dict):
                    bad += 1
                    print(f"[CLEAN] {src}:{lineno} — not a dict; dropped")
                    continue
                fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
                good += 1
            except json.JSONDecodeError as e:
                bad += 1
                print(f"[CLEAN] {src}:{lineno} — bad JSON: {e}; dropped")
    print(f"[CLEAN] wrote {good} valid lines to {dst}; dropped {bad} bad lines")

# ------------------------------------------------------------
# Main process
# ------------------------------------------------------------

def main():
    episodes = read_jsonl(INPUT_JSONL)
    done = already_done_index(OUT_PLOTS_JSONL)
    print(f"Loaded {len(episodes)} episode entries. Already have {len(done)} plot records logged.")

    n_ok = 0
    for i, ep in enumerate(episodes, 1):
        url = ep.get("title_link", "")
        title = ep.get("title", "")

        if not url:
            print(f"[{i}/{len(episodes)}] {title} — skipped (no URL)")
            continue

        prev = done.get(url)
        if not FORCE and prev and prev.get("status") == "ok":
            print(f"[{i}/{len(episodes)}] {title} — cached ok")
            n_ok += 1
            continue

        time.sleep(REQUEST_DELAY)
        plot = fetch_plot(url)
        status = "ok" if plot else "missing"
        rec = {
            "title": title,
            "title_link": url,
            "plot_text": plot,
            "status": status,
            "source": "Wikipedia (CC BY-SA 4.0)"
        }
        append_jsonl(OUT_PLOTS_JSONL, rec)
        if status == "ok":
            n_ok += 1
        print(f"[{i}/{len(episodes)}] {title} — {status}")

    # Build merged CSV using the latest record per URL
    plots = {r["title_link"]: r for r in read_jsonl(OUT_PLOTS_JSONL)}
    merged = []
    for ep in episodes:
        url = ep.get("title_link", "")
        p = plots.get(url, {})
        merged.append({
            "season": ep.get("season", ""),
            "overall_no": ep.get("overall_no", ""),
            "season_no": ep.get("season_no", ""),
            "title": ep.get("title", ""),
            "title_link": url,
            "original_air_date": ep.get("original_air_date", ""),
            "short_summary": ep.get("short_summary", ""),
            "plot_text": p.get("plot_text", ""),
            "plot_status": p.get("status", "missing"),
            "source_page": ep.get("source_page", ""),
        })

    if merged:
        with open(OUT_MERGED_CSV, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(merged[0].keys()))
            w.writeheader()
            w.writerows(merged)

    print(f"\nDone. Plots OK: {n_ok} / {len(episodes)}")
    print(f"Wrote: {OUT_PLOTS_JSONL}")
    print(f"Wrote: {OUT_MERGED_CSV}")
    print('\nAttribution:')
    print('Text derived from Wikipedia ("List of Seinfeld episodes" and episode pages), CC BY-SA 4.0. '
          '© Wikipedia contributors. Source: https://en.wikipedia.org/wiki/List_of_Seinfeld_episodes')

# ------------------------------------------------------------
if __name__ == "__main__":
    main()


[WARN] seinfeld_plots.jsonl:1 — bad JSON: Expecting ',' delimiter: line 1 column 658 (char 657). Skipped.
Loaded 180 episode entries. Already have 170 plot records logged.
[1/180] The Seinfeld Chronicles — missing
[2/180] The Stake Out — cached ok
[3/180] The Robbery — cached ok
[4/180] Male Unbonding — cached ok
[5/180] The Stock Tip — cached ok
[6/180] The Ex-Girlfriend — cached ok
[7/180] The Pony Remark — cached ok
[8/180] The Jacket — cached ok
[9/180] The Phone Message — cached ok
[10/180] The Apartment — cached ok
[11/180] The Statue — cached ok
[12/180] The Revenge — cached ok
[13/180] The Heart Attack — cached ok
[14/180] The Deal — cached ok
[15/180] The Baby Shower — cached ok
[16/180] The Chinese Restaurant — cached ok
[17/180] The Busboy — cached ok
[18/180] The Note — cached ok
[19/180] The Truth — cached ok
[20/180] The Pen — cached ok
[21/180] The Dog — cached ok
[22/180] The Library — cached ok
[23/180] The Parking Garage — cached ok
[24/180] The Café — cached ok
[25/1